# classifier_attribution — Colab
Attribution par CLASSIFIEUR (empreinte de perplexite).
Rapide, **pas besoin de GPU**. Utilise les 2 fichiers produits par `detector.ipynb`.

In [1]:
# 1) Dependances
!pip -q install scikit-learn pandas numpy

## 2) Charge les 2 fichiers
Selectionne **dataset_scored.csv** ET **perplexity_matrix.npy**.

In [2]:
from google.colab import files
up = files.upload()   # -> dataset_scored.csv + perplexity_matrix.npy

Saving dataset_scored.csv to dataset_scored.csv
Saving perplexity_matrix.npy to perplexity_matrix.npy


## 3) Le code (features + classifieurs + analyse d'erreurs)

In [3]:
"""
classifier_attribution.py
------------------------------------------------------------------
Attribution par CLASSIFIEUR, base sur l'"empreinte de perplexite".

Idee : on ne peut pas calculer la perplexite des modeles FERMES (GPT, Gemini),
mais on peut representer CHAQUE texte (open OU ferme) par son empreinte =
le vecteur de ses perplexites sous nos modeles open de reference.
Un classifieur apprend alors a relier cette empreinte a l'auteur reel,
y compris les modeles fermes -> ils entrent enfin dans l'attribution.

On reutilise les sorties de detector.py :
  - dataset_scored.csv     (memes lignes, meme ordre)
  - perplexity_matrix.npy  (perplexite de chaque texte sous chaque modele open)

Dependances :
  pip install pandas numpy scikit-learn
------------------------------------------------------------------
"""

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)

# ============================================================
# CONFIG
# ============================================================
SCORED_CSV = "dataset_scored.csv"        # produit par detector.py
PPL_MATRIX = "perplexity_matrix.npy"     # produit par detector.py

# "all"          -> une seule classe par source : human + chaque modele
#                   (fait detection ET attribution d'un coup)
# "models_only"  -> uniquement entre modeles (on retire les textes humains)
TARGET = "all"

USE_BINOCULARS_FEATURE = True   # ajoute le score Binoculars comme feature en plus


# ============================================================
# CHARGEMENT + FEATURES
# ============================================================
def load_data():
    df = pd.read_csv(SCORED_CSV)
    ppl = np.load(PPL_MATRIX)                       # [n_textes, n_modeles_open]
    assert len(df) == ppl.shape[0], \
        "dataset_scored.csv et perplexity_matrix.npy ne sont pas alignes !"

    # impute les NaN eventuels (un modele qui aurait echoue) par la moyenne colonne
    col_mean = np.nanmean(ppl, axis=0)
    inds = np.where(np.isnan(ppl))
    ppl[inds] = np.take(col_mean, inds[1])

    # L'EMPREINTE = les perplexites sous les modeles open de reference
    X = ppl.copy()

    # option : ajoute le score Binoculars comme signal supplementaire
    if USE_BINOCULARS_FEATURE and "binoculars_score" in df.columns:
        X = np.column_stack([X, df["binoculars_score"].values])

    # label = l'auteur (human ou nom du modele)
    y = df["model_name"].astype(str).values
    return df, X, y


# ============================================================
# ENTRAINEMENT + EVALUATION
# ============================================================
def run(df, X, y):
    if TARGET == "models_only":
        keep = df["source_type"].eq("model").values
        df, X, y = df[keep], X[keep], y[keep]
        df = df.reset_index(drop=True)

    tr = df["split"].eq("train").values
    te = df["split"].eq("test").values

    # standardisation des features (ajustee sur le train uniquement)
    scaler = StandardScaler().fit(X[tr])
    Xtr, Xte = scaler.transform(X[tr]), scaler.transform(X[te])

    n_classes = len(np.unique(y))
    print(f"Classes ({n_classes}) :", sorted(np.unique(y)))
    print(f"Hasard = {1/n_classes:.3f}   |   train={tr.sum()}  test={te.sum()}\n")

    models = {
        "LogisticRegression": LogisticRegression(max_iter=2000, C=1.0),
        "RandomForest":       RandomForestClassifier(n_estimators=300, random_state=42),
    }

    for name, clf in models.items():
        clf.fit(Xtr, y[tr])
        pred = clf.predict(Xte)
        acc = accuracy_score(y[te], pred)
        f1  = f1_score(y[te], pred, average="macro")
        print(f"===== {name} =====")
        print(f"accuracy={acc:.3f}   macro-F1={f1:.3f}")
        print(classification_report(y[te], pred, zero_division=0))
        labels = sorted(np.unique(y))
        print("Matrice de confusion (lignes=vrai, cols=predit), ordre =", labels)
        print(confusion_matrix(y[te], pred, labels=labels))
        print()

        # sauvegarde des predictions du meilleur modele lisible
        out = df[te][["id", "domain", "source_type", "model_name"]].copy()
        out["predicted"] = pred
        out.to_csv(f"attribution_pred_{name}.csv", index=False)

        # --- analyse des erreurs : ou et comment le classifieur se trompe ---
        err = out[out["model_name"] != out["predicted"]]
        print(f"Erreurs : {len(err)} / {len(out)} textes de test")
        if len(err):
            print(err[["id", "domain", "model_name", "predicted"]].to_string(index=False))
            print("\nErreurs par auteur reel :")
            print(err["model_name"].value_counts().to_string())
        print()


def main():
    df, X, y = load_data()
    run(df, X, y)
    print("OK -> predictions sauvegardees (attribution_pred_*.csv)")


## 4) Lance l'attribution
Affiche accuracy / macro-F1 / matrices de confusion / erreurs, puis telecharge les predictions.

In [4]:
main()

from google.colab import files
files.download('attribution_pred_LogisticRegression.csv')
files.download('attribution_pred_RandomForest.csv')

Classes (6) : ['Mistral-7B-Instruct-v0.3', 'Phi-4-mini-instruct', 'Qwen3-4B-Instruct-2507', 'SmolLM3-3B', 'gemma-4-E4B-it', 'human']
Hasard = 0.167   |   train=200  test=50

===== LogisticRegression =====
accuracy=0.920   macro-F1=0.864
                          precision    recall  f1-score   support

Mistral-7B-Instruct-v0.3       1.00      1.00      1.00         5
     Phi-4-mini-instruct       1.00      1.00      1.00         5
  Qwen3-4B-Instruct-2507       0.83      1.00      0.91         5
              SmolLM3-3B       1.00      1.00      1.00         5
          gemma-4-E4B-it       1.00      0.20      0.33         5
                   human       0.89      1.00      0.94        25

                accuracy                           0.92        50
               macro avg       0.95      0.87      0.86        50
            weighted avg       0.93      0.92      0.90        50

Matrice de confusion (lignes=vrai, cols=predit), ordre = ['Mistral-7B-Instruct-v0.3', 'Phi-4-mini-in

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>